# Task 3: Vectorized Convolutional Forward Pass via im2col

## Objective

Implement a 2D convolutional layer from scratch using the **im2col transformation** and vectorized matrix multiplication.

The implementation supports:

- Arbitrary input/output channels
- Arbitrary kernel height and width
- Arbitrary padding
- Arbitrary stride
- Batch processing
- im2col transformation
- Vectorized convolution using matrix multiplication
- Verification using SciPy

In [ ]:
import numpy as np
from scipy import signal

np.random.seed(42)

# Example configuration
N = 2
C_in = 3
H, W = 8, 8

C_out = 4
K_h, K_w = 3, 3

stride = (2, 2)
padding = (1, 1)

# Input
X = np.random.randn(
    N, C_in, H, W
)

# Convolution kernels
K = np.random.randn(
    C_out,
    C_in,
    K_h,
    K_w
)

# Bias
b = np.random.randn(C_out)

print("Input shape :", X.shape)
print("Kernel shape:", K.shape)
print("Bias shape  :", b.shape)

# im2col Transformation

The im2col operation converts every sliding convolution window into one row of a matrix.

For each output location, the corresponding:

\[
C_{in}\times K_h\times K_w
\]

values are flattened into one vector.

Therefore, all convolution windows can be processed simultaneously using matrix multiplication instead of nested convolution loops.

In [ ]:
def im2col(X, kernel_size, stride=(1, 1), padding=(0, 0)):
    """
    Convert convolution sliding windows into a 2D matrix.

    X:
        (N, C, H, W)

    Returns:
        cols:
            (N * H_out * W_out, C * Kh * Kw)
        H_out, W_out
    """

    N, C, H, W = X.shape
    Kh, Kw = kernel_size

    Sh, Sw = stride
    Ph, Pw = padding

    # Pad input
    X_pad = np.pad(
        X,
        (
            (0, 0),
            (0, 0),
            (Ph, Ph),
            (Pw, Pw)
        ),
        mode="constant"
    )

    H_pad, W_pad = X_pad.shape[2:]

    # Output dimensions
    H_out = (
        (H_pad - Kh) // Sh
    ) + 1

    W_out = (
        (W_pad - Kw) // Sw
    ) + 1

    # Build sliding windows
    windows = np.lib.stride_tricks.sliding_window_view(
        X_pad,
        (Kh, Kw),
        axis=(2, 3)
    )

    # Apply stride
    windows = windows[
        :,
        :,
        ::Sh,
        ::Sw,
        :,
        :
    ]

    # (N,C,H_out,W_out,Kh,Kw)
    # -> (N,H_out,W_out,C,Kh,Kw)
    windows = windows.transpose(
        0, 2, 3, 1, 4, 5
    )

    # Flatten each window
    cols = windows.reshape(
        N * H_out * W_out,
        C * Kh * Kw
    )

    return cols, H_out, W_out


def conv2d_im2col(
    X,
    K,
    b=None,
    stride=(1, 1),
    padding=(0, 0)
):
    """
    Vectorized 2D convolution using im2col.
    """

    N, C_in, H, W = X.shape

    C_out, _, Kh, Kw = K.shape

    # Convert input windows to columns
    X_col, H_out, W_out = im2col(
        X,
        (Kh, Kw),
        stride,
        padding
    )

    # Flatten kernels
    K_col = K.reshape(
        C_out,
        -1
    )

    # Single vectorized matrix multiplication
    output_col = X_col @ K_col.T

    if b is not None:
        output_col += b

    # Restore NCHW format
    output = output_col.reshape(
        N,
        H_out,
        W_out,
        C_out
    ).transpose(
        0, 3, 1, 2
    )

    return output, X_col


# Run im2col convolution
output, X_col = conv2d_im2col(
    X,
    K,
    b,
    stride=stride,
    padding=padding
)

print("im2col shape :", X_col.shape)
print("Output shape :", output.shape)

# Vectorized Convolution and SciPy Verification

The im2col implementation is verified against SciPy's multidimensional cross-correlation.

Deep-learning convolution layers commonly implement the cross-correlation operation, meaning the kernel is not flipped.

The outputs should match up to floating-point precision.

In [ ]:
# ============================================================
# SciPy verification
# ============================================================

def scipy_conv2d(
    X,
    K,
    b=None,
    stride=(1, 1),
    padding=(0, 0)
):

    N, C_in, H, W = X.shape
    C_out, _, Kh, Kw = K.shape

    Sh, Sw = stride
    Ph, Pw = padding

    # Pad input
    X_pad = np.pad(
        X,
        (
            (0, 0),
            (0, 0),
            (Ph, Ph),
            (Pw, Pw)
        ),
        mode="constant"
    )

    H_pad, W_pad = X_pad.shape[2:]

    H_out = (
        (H_pad - Kh) // Sh
    ) + 1

    W_out = (
        (W_pad - Kw) // Sw
    ) + 1

    result = np.zeros(
        (N, C_out, H_out, W_out)
    )

    # SciPy is used only for verification
    for n in range(N):

        for cout in range(C_out):

            channel_result = np.zeros(
                (H_pad - Kh + 1,
                 W_pad - Kw + 1)
            )

            for cin in range(C_in):

                channel_result += signal.correlate2d(
                    X_pad[n, cin],
                    K[cout, cin],
                    mode="valid"
                )

            result[
                n,
                cout
            ] = channel_result[
                ::Sh,
                ::Sw
            ]

            if b is not None:
                result[
                    n,
                    cout
                ] += b[cout]

    return result


scipy_output = scipy_conv2d(
    X,
    K,
    b,
    stride=stride,
    padding=padding
)

difference = np.max(
    np.abs(
        output - scipy_output
    )
)

print("im2col output shape :", output.shape)
print("SciPy output shape  :", scipy_output.shape)
print("Maximum difference  :", difference)

if np.allclose(
    output,
    scipy_output,
    atol=1e-10
):
    print("Verification: PASSED")
else:
    print("Verification: FAILED")

# Experiment with Different Configurations

The convolution implementation supports arbitrary:

- Input channels
- Output channels
- Kernel dimensions
- Padding
- Stride

The following examples demonstrate that the same implementation works with different convolution configurations.

## Conclusion

The 2D convolutional forward pass was implemented from scratch using the im2col transformation.

The sliding windows were converted into a unified 2D matrix of shape:

\[
(NH_{out}W_{out})
\times
(C_{in}K_hK_w)
\]

The kernels were reshaped into:

\[
C_{out}
\times
(C_{in}K_hK_w)
\]

and convolution was reduced to the matrix multiplication:

\[
Y_{col}=X_{col}K_{col}^{T}
\]

The resulting matrix was reshaped back into the standard convolutional output format:

\[
N\times C_{out}\times H_{out}\times W_{out}
\]

The implementation was also verified against SciPy, confirming the correctness of the vectorized convolution operation.

In [ ]:
# ============================================================
# Additional configurations
# ============================================================

configs = [
    {
        "kernel": (3, 3),
        "stride": (1, 1),
        "padding": (0, 0)
    },
    {
        "kernel": (3, 3),
        "stride": (2, 2),
        "padding": (1, 1)
    },
    {
        "kernel": (5, 3),
        "stride": (1, 2),
        "padding": (2, 1)
    }
]

for config in configs:

    kh, kw = config["kernel"]

    K_test = np.random.randn(
        C_out,
        C_in,
        kh,
        kw
    )

    result, columns = conv2d_im2col(
        X,
        K_test,
        stride=config["stride"],
        padding=config["padding"]
    )

    print(
        f"Kernel={config['kernel']}, "
        f"Stride={config['stride']}, "
        f"Padding={config['padding']} "
        f"-> Output={result.shape}, "
        f"im2col={columns.shape}"
    )

# Conclusion

The 2D convolutional forward pass was successfully implemented from scratch using the **im2col transformation** and NumPy.

The implementation supports arbitrary:

- Input and output channels
- Kernel height and width
- Stride
- Padding
- Batch size

The sliding convolution windows were transformed into a 2D matrix:

\[
X_{col}
\in
\mathbb{R}^{(N H_{out}W_{out})\times(C_{in}K_hK_w)}
\]

The convolution was then performed efficiently using a single matrix multiplication:

\[
Y_{col}=X_{col}K_{col}^{T}
\]

The resulting matrix was reshaped back into the standard 4D convolution output:

\[
(N,C_{out},H_{out},W_{out})
\]

The implementation was verified against SciPy, and the negligible numerical difference confirms that the vectorized im2col convolution produces the correct results.

Thus, this experiment demonstrates how convolution operations can be transformed into efficient matrix multiplications suitable for high-performance computation.